In this notebook I want to compute weighted phase lag index

In [ ]:
import numpy as np
import mne
import mne_connectivity
import tensorpac
from tensorpac import Pac
from tensorpac.signals import pac_signals_wavelet
from mne_connectivity import phase_slope_index, seed_target_indices, spectral_connectivity_epochs
from mne.stats import permutation_cluster_test, permutation_cluster_1samp_test
from mne.filter import next_fast_len
from mne.preprocessing.bads import _find_outliers

from glob import glob
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
import seaborn as sns
from scipy.stats import zscore, linregress, ttest_ind, ttest_rel, ttest_1samp, pearsonr
import pandas as pd
from scipy.signal import hilbert, correlate, correlation_lags
import fooof
# Import plotting functions
from fooof.plts.spectra import plot_spectrum, plot_spectra
# Import the FOOOF object
from fooof import FOOOF
from fooof import FOOOFGroup
import os 
import joblib
import statsmodels.formula.api as smf
import statsmodels.api as sm
import pickle
from tqdm import tqdm
from IPython.display import clear_output
import warnings 
from ast import literal_eval


# I only want to see warnings once
warnings.filterwarnings('ignore')

from joblib import delayed, Parallel
from statsmodels.stats import multitest

In [ ]:
import sys
sys.path.append('/hpc/users/qasims01/resources/LFPAnalysis')

In [ ]:
from LFPAnalysis import lfp_preprocess_utils, sync_utils, analysis_utils, nlx_utils, oscillation_utils, statistics_utils


In [ ]:
base_dir = '/sc/arion' # this is the root directory for most un-archived data and results
load_dir = f'{base_dir}/work/qasims01/MemoryBanditData/EMU'

# Load the demographics data to get information about who has run the task
demo_file = f'{base_dir}/projects/guLab/Salman/EMU/MemoryBanditsDatabase.xlsx'
demo_data = pd.read_excel(demo_file, engine='openpyxl')
subj_df = demo_data[(demo_data.LongVersion==1) & (demo_data.Minerva==1)]

# Let's get rid of Iowa patients for now 

# subj_df = subj_df[subj_df.Site=='UI']
subj_df = subj_df[~subj_df['MSSMCode'].isin(['MS015', 'MS024', 'MS034', 'MS038'])]


days = ['Day1', 'Day2']
subj_ids = subj_df.MSSMCode.values
subj_formats = subj_df.format.values
subj_sites = subj_df.Site.values
subj_days = subj_df.Day2.values

In [ ]:
# load behavioral data
learn_df = pd.read_csv(f'{load_dir}/learn_df_RWH.csv')
combined_df = pd.read_csv(f'{load_dir}/full_df_RWH.csv')

In [ ]:
# Explicitly define a list of analysis events and the baseline event. Should correspond to the dict

evs = {'gamble_start':[-1.0, 0], 
       'feedback_start': [0, 1.5],
       'baseline_start': [0, 0.75]}

baseline_ev = 'baseline_start'
analysis_evs = ['feedback_start']

day = 'Day1'

for i in analysis_evs: 
    if i not in evs.keys(): 
        raise ValueError('This analysis event is not in the dictionary')

In [ ]:
# From Bahramsharif 2013
# get where phase frequencies start / finish / number
phase_frequency = 'theta'
f_pha_start, f_pha_end = 2, 8
f_pha_nb = 30
f_pha_mid = np.linspace(f_pha_start, f_pha_end, f_pha_nb)
f_pha = np.c_[f_pha_mid - f_pha_mid / 4., f_pha_mid + f_pha_mid / 4.]
# get where amplitude frequencies start / finish / number
amp_frequency = 'hfa'

f_amp_start, f_amp_end = 70, 200
f_amp_nb = 30
f_amp_mid = np.linspace(f_amp_start, f_amp_end, f_amp_nb)
f_amp = np.c_[f_amp_mid - f_amp_mid / 8., f_amp_mid + f_amp_mid / 8.]

Within-region PAC

In [ ]:
base_dir = '/sc/arion'
save_dir_plots = '/sc/arion/work/qasims01/MemoryBanditData/EMU/plots'
save_dir_data = '/sc/arion/work/qasims01/MemoryBanditData/EMU/dataframes'
# 
n_perm = 100  # number of permutations
# define the Pac object (gaussian-copula PAC) 
p = Pac(idpac=(6, 2, 0), f_pha=f_pha, f_amp=f_amp, dcomplex='wavelet')

day = 'Day1'

for roi in ['OFC']:
    all_dfs = [] 
    for subj_id in subj_ids:

        event = 'feedback_start'
        baseline_event = 'baseline_start'
        # baseline_start_mem
        loadpath = f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj_id}'
        save_path = f'/sc/arion/projects/guLab/Salman/EphysAnalyses/{subj_id}/neural/{day}'

        # For each subject, load the epochs of interest and the baseline epoch, as well as the electrode dataframe 
        epochs_reref = mne.read_epochs(f'{save_path}/{event}-epo.fif',  preload=True) 

        elec_df = pd.read_csv(f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj_id}/Day1_reref_elec_df')

        buf_ms = 1350 # special case (10/17) - I want to isolate a  specific window in the OFC 
        edges = buf_ms * (epochs_reref.info['sfreq']/1000)

        # compute true pac and surrogates
        elecs = np.where(elec_df.salman_region == roi)[0] 

        pwise_dfs = []
        if len(elecs)>0:
            for ei in elecs:
                # ch_name = epochs_reref.ch_names[ei]
                ch_name = elec_df.loc[ei, 'label']

                real_sig = np.squeeze(epochs_reref.copy().pick(ch_name)._data)
                # if doing cross-regional pac
                # amp_sig = np.squeeze(epochs_reref.copy().pick(['rmolf1-rmolf2'])._data) 
                
                # if os.path.isfile(f'{save_dir_data}/{subj_id}_{roi}_{ch_name}_pac.npy'):
                #     print('already ran this one')
                #     continue

                epoch_pac = p.filterfit(epochs_reref.info['sfreq'], 
                                 real_sig, 
                                 n_perm=n_perm, n_jobs=-1,
                                 edges=edges, random_state=0)


                surr_mean = p.surrogates.mean(axis=0)
                surr_std = p.surrogates.std(axis=0)
                z_scored_pac = (epoch_pac-surr_mean)/surr_std

                # Save out the z_scored pac so I can do individual analysis on it
                np.save(f'{save_dir_data}/{subj_id}_{roi}_{ch_name}_pac', z_scored_pac)

                # average the z-scores across phae and frequency amplitudes 
                
                avg_conn = z_scored_pac.mean(axis=(0, 1))
                
                # alternative: nan out non-significant pac_values, then sum
                z_scored_pac[np.abs(z_scored_pac)<=1.96] = np.nan
                
                sum_conn = np.nansum(z_scored_pac, axis=(0, 1))

                pwise_df = pd.DataFrame(columns=['participant', 'roi', 'label', 'event', 
                                                 'avg_conn', 'sum_conn', 'hemi'])

                pwise_df['avg_conn'] = avg_conn
                pwise_df['sum_conn'] = sum_conn
                pwise_df['hemi'] = elec_df[elec_df.label==ch_name].hemisphere.values[0]
                pwise_df['participant'] = subj_id
                pwise_df['age'] = subj_df[subj_df.MSSMCode==subj_id].Age.values[0]
                pwise_df['sex'] = subj_df[subj_df.MSSMCode==subj_id].Sex.str.strip().values[0]

                pwise_df['roi'] = roi
                pwise_df['event'] = event
                pwise_df['band0'] = phase_frequency
                pwise_df['band1'] = amp_frequency
                pwise_df['trials'] = np.arange(1, len(avg_conn)+1)
                pwise_df['label'] = ch_name


                pwise_df = pwise_df.merge(learn_df[learn_df.participant==subj_id], on=['participant', 'trials'])

                pwise_df['zrpe'] = (pwise_df.rpe - np.nanmean(pwise_df.rpe)) / (2*np.nanstd(pwise_df.rpe))
                pwise_df['zpm'] = (pwise_df.DPRIME - np.nanmean(pwise_df.DPRIME)) / (2*np.nanstd(pwise_df.DPRIME))

    #             # aggregate
                pwise_dfs.append(pwise_df)
        else:
            # this subject does not have electrodes in both regions
            continue
        
        if len(pwise_dfs) > 0:
            all_df_pt = pd.concat(pwise_dfs)
            all_df_pt.to_csv(f'{save_dir_data}/{subj_id}_{roi}_pac_{event}.csv', index=False)
            all_dfs.append(all_df_pt)

    all_dfs = pd.concat(all_dfs)
    all_dfs.to_csv(f'{save_dir_data}/all_{roi}_{phase_frequency}_pac_{event}.csv', index=False)


In [ ]:
base_dir = '/sc/arion'
save_dir_plots = '/sc/arion/work/qasims01/MemoryBanditData/EMU/plots'
save_dir_data = '/sc/arion/work/qasims01/MemoryBanditData/EMU/dataframes'
# 
n_perm = 500  # number of permutations
# define the Pac object (gaussian-copula PAC) 
p = Pac(idpac=(6, 2, 0), f_pha=f_pha, f_amp=f_amp, dcomplex='wavelet')

day = 'Day1'

for roi in ['OFC']:
    all_dfs = [] 
    for subj_id in subj_ids:

        event = 'feedback_start'
        baseline_event = 'baseline_start'
        # baseline_start_mem
        loadpath = f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj_id}'
        save_path = f'/sc/arion/projects/guLab/Salman/EphysAnalyses/{subj_id}/neural/{day}'

        # For each subject, load the epochs of interest and the baseline epoch, as well as the electrode dataframe 
        epochs_reref = mne.read_epochs(f'{save_path}/{event}-epo.fif',  preload=True) 

        elec_df = pd.read_csv(f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj_id}/Day1_reref_elec_df')

        buf_ms = 1000
        edges = buf_ms * (epochs_reref.info['sfreq']/1000)

        # compute true pac and surrogates
        elecs = np.where(elec_df.salman_region == roi)[0] 

        pwise_dfs = []
        if len(elecs)>0:
            for ei in elecs:
                # ch_name = epochs_reref.ch_names[ei]
                ch_name = elec_df.loc[ei, 'label']
                
                z_scored_pac = np.load(f'{save_dir_data}/{subj_id}_{roi}_{ch_name}_pac.npy')
                # average the z-scores across phae and frequency amplitudes 
                
                avg_conn = z_scored_pac.mean(axis=(0, 1))
                
                # alternative: nan out non-significant pac_values, then sum
                z_scored_pac[np.abs(z_scored_pac)<=1.96] = np.nan
                
                sum_conn = np.nansum(z_scored_pac, axis=(0, 1))

                pwise_df = pd.DataFrame(columns=['participant', 'roi', 'label', 'event', 
                                                 'avg_conn', 'sum_conn', 'hemi'])

                pwise_df['avg_conn'] = avg_conn
                pwise_df['sum_conn'] = sum_conn
                pwise_df['hemi'] = elec_df[elec_df.label==ch_name].hemisphere.values[0]
                pwise_df['participant'] = subj_id
                pwise_df['age'] = subj_df[subj_df.MSSMCode==subj_id].Age.values[0]
                pwise_df['sex'] = subj_df[subj_df.MSSMCode==subj_id].Sex.str.strip().values[0]

                pwise_df['roi'] = roi
                pwise_df['event'] = event
                pwise_df['band0'] = phase_frequency
                pwise_df['band1'] = amp_frequency
                pwise_df['trials'] = np.arange(1, len(avg_conn)+1)
                pwise_df['label'] = ch_name


                pwise_df = pwise_df.merge(learn_df[learn_df.participant==subj_id], on=['participant', 'trials'])

                pwise_df['zrpe'] = (pwise_df.rpe - np.nanmean(pwise_df.rpe)) / (2*np.nanstd(pwise_df.rpe))
                pwise_df['zpm'] = (pwise_df.DPRIME - np.nanmean(pwise_df.DPRIME)) / (2*np.nanstd(pwise_df.DPRIME))

    #             # aggregate
                pwise_dfs.append(pwise_df)
        else:
            # this subject does not have electrodes in both regions
            continue
        
        if len(pwise_dfs) > 0:
            all_df_pt = pd.concat(pwise_dfs)
            all_df_pt.to_csv(f'{save_dir_data}/{subj_id}_{roi}_pac_{event}.csv', index=False)
            all_dfs.append(all_df_pt)

    all_dfs = pd.concat(all_dfs)
    all_dfs.to_csv(f'{save_dir_data}/all_{roi}_pac_{event}.csv', index=False)


In [ ]:
# base_dir = '/sc/arion'
# save_dir_plots = '/sc/arion/work/qasims01/MemoryBanditData/EMU/plots'
# save_dir_data = '/sc/arion/work/qasims01/MemoryBanditData/EMU/dataframes'
# # 
# n_perm = 500  # number of permutations
# # define the Pac object (gaussian-copula PAC) 
# p = Pac(idpac=(6, 2, 0), f_pha=f_pha, f_amp=f_amp, dcomplex='wavelet')

# day = 'Day1'

# all_dfs = [] 

# for roi in ['OFC']:
#     for subj_id in subj_ids:

#         event = 'feedback_start'
#         baseline_event = 'baseline_start'
#         # baseline_start_mem
#         loadpath = f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj_id}'
#         save_path = f'/sc/arion/projects/guLab/Salman/EphysAnalyses/{subj_id}/neural/{day}'

#         elec_df = pd.read_csv(f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj_id}/Day1_reref_elec_df')
#         epochs_reref = mne.read_epochs(f'{save_path}/{event}-epo.fif',  preload=True) 
#         ch_names = epochs_reref.ch_names
#         del epochs_reref

#         elecs = np.where(elec_df.salman_region == roi)[0]            

#         pwise_dfs = []
#         if len(elecs)>0:
#             for ei in elecs:
#                 ch_name = ch_names[ei]
#                 z_scored_pac = np.load(f'{save_dir_data}/{subj_id}_{roi}_{ch_name}_pac.npy')
#                 # f = plt.figure(figsize=(11, 5))
#                 # plt.subplot(121)
#                 # p.comodulogram(z_scored_pac, title=str(p), cmap='Spectral_r')
                
                
#                 # # average the z-scores across phae and frequency amplitudes
#                 avg_conn = z_scored_pac.mean(axis=(0, 1))
                
#                 # alternative: nan out non-significant pac_values, then sum
#                 z_scored_pac[np.abs(z_scored_pac)<=1.96] = np.nan
                
#                 sum_conn = np.nansum(z_scored_pac, axis=(0, 1))

#                 pwise_df = pd.DataFrame(columns=['participant', 'roi', 'label', 'event', 'conn'])

#                 pwise_df['avg_conn'] = avg_conn
#                 pwise_df['sum_conn'] = sum_conn
#                 pwise_df['hemi'] = elec_df[elec_df.label==ch_name].hemisphere.values[0]
#                 pwise_df['participant'] = subj_id
#                 pwise_df['age'] = subj_df[subj_df.MSSMCode==subj_id].Age.values[0]
#                 pwise_df['sex'] = subj_df[subj_df.MSSMCode==subj_id].Sex.str.strip().values[0]

#                 pwise_df['roi'] = roi
#                 pwise_df['event'] = event
#                 pwise_df['band0'] = phase_frequency
#                 pwise_df['band1'] = amp_frequency
#                 pwise_df['trials'] = np.arange(1, len(conn)+1)
#                 pwise_df['label'] = ch_name


#                 pwise_df = pwise_df.merge(learn_df[learn_df.participant==subj_id], on=['participant', 'trials'])

#                 pwise_df['zrpe'] = (pwise_df.rpe - np.nanmean(pwise_df.rpe)) / (2*np.nanstd(pwise_df.rpe))
#                 pwise_df['zpm'] = (pwise_df.DPRIME - np.nanmean(pwise_df.DPRIME)) / (2*np.nanstd(pwise_df.DPRIME))

#     #             # aggregate
#                 pwise_dfs.append(pwise_df)
#         else:
#             # this subject does not have electrodes in both regions
#             continue
            
#         all_df_pt = pd.concat(pwise_dfs)
#         all_df_pt.to_csv(f'{save_dir_data}/{subj_id}_{roi}_pac_{event}.csv', index=False)
#         all_dfs.append(all_df_pt)

#     all_dfs = pd.concat(all_dfs)
#     all_dfs.to_csv(f'{save_dir_data}/all_{roi}_pac_{event}.csv', index=False)


In [ ]:
elec_df[elec_df.label==ch_name].hemisphere.values[0]

In [ ]:
for trial in range(5):
    f = plt.figure(figsize=(11, 5))
    plt.subplot(121)
    plot_pac = z_scored_pac[:, :, trial].copy()
    plot_pac[np.abs(plot_pac) <= 1.96] = np.nan
    p.comodulogram(plot_pac, title=str(p), cmap='Spectral_r')

In [ ]:
# f = plt.figure(figsize=(11, 5))
# plt.subplot(121)
# p.comodulogram(z_scored_pac, title=str(p), cmap='Spectral_r')


# t_obs, clusters, cluster_p_values, h0 = permutation_cluster_1samp_test(np.moveaxis(z_scored_pac, -1, 0), 
#                                                                        n_permutations=1000, n_jobs=-1, tail=1)
# # # split two conditions:
# # p.comodulogram(z_scored_pac[:, :, pwise_df.rpe<0], title=str(p), cmap='Spectral_r')
# # plt.subplot(122)
# # p.comodulogram(z_scored_pac[:, :, pwise_df.rpe>0], title=str(p), cmap='Spectral_r')
# # t_obs, clusters, cluster_p_values, h0 = permutation_cluster_test([np.moveaxis(z_scored_pac[:, :, pwise_df.rpe>0], -1, 0),
# #                                                                   np.moveaxis(z_scored_pac[:, :, pwise_df.rpe<0], -1, 0)],
# #                                                                        n_permutations=500, n_jobs=-1, tail=1)
# # # if len(cluster_p_values) > 0:
# #     if any(cluster_p_values <= 0.05):
# #         f = plt.figure(figsize=(11, 5))
# #         plt.subplot(121)
# #         p.comodulogram(z_scored_pac, title=str(p), cmap='Spectral_r')


# #         # create new stats image with only significant clusters
# #         t_obs_plot = np.nan * np.ones_like(t_obs)
# #         t_obs_background = t_obs.copy()
# #         for c, p_val in zip(clusters, cluster_p_values):
# #             if p_val <= 0.05:
# #                 t_obs_plot[c] = t_obs[c]
# #                 t_obs_background[c] = np.nan

In [ ]:
f = plt.figure(figsize=(11, 5))
plt.subplot(121)
p.comodulogram(z_scored_pac[:, :, pwise_df.rpe>0], title=str(p), cmap='Spectral_r')


# create new stats image with only significant clusters
t_obs_plot = np.nan * np.ones_like(t_obs)
t_obs_background = t_obs.copy()
for c, p_val in zip(clusters, cluster_p_values):
    if p_val <= 0.15:
        t_obs_plot[c] = t_obs[c]
        t_obs_background[c] = np.nan
plt.subplot(122)
p.comodulogram(t_obs_background, cmap='gray', colorbar=False)
p.comodulogram(t_obs_plot, title='Permutation-tested PAC (against z=0)', cmap='viridis', vmin=0, vmax=np.max(t_obs))
plt.gca().invert_yaxis()
plt.ylabel('')

plt.tight_layout()

In [ ]:
cluster_p_values

In [ ]:
cluster_p_values

In [ ]:
save_dir_data = '/sc/arion/work/qasims01/MemoryBanditData/EMU/dataframes'
roi = 'dmPFC'
event = 'feedback_start'
all_dfs = pd.read_csv(f'{save_dir_data}/all_{roi}_pac_{event}.csv')

In [ ]:

all_dfs = pd.read_csv(f'{save_dir_data}/all_AMY_OFC_pac_feedback_start.csv')


In [ ]:
all_dfs

In [ ]:
test_model = smf.mixedlm(formula='conn ~ 1 + zrpe + zpm',
                         re_formula = '1 + zrpe + zpm', 
                         data=all_dfs, 
                         groups=all_dfs['label']).fit()


In [ ]:
test_model.summary()

In [ ]:
sns.regplot(data=all_dfs, x='rpe', y='conn', scatter=False)

Cross-regional PAC (need to debug this)